In [11]:
import sys
sys.path.insert(1, '/home/bricej/MyPythonLibrary/StageM2_IVT/library/')
import domain
from domain import *
from climbas import *
import numpy as np
import geopandas as gpd
import xarray as xr
import pandas as pd
from netCDF4 import Dataset
import dask
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib as mpl
from shapely.geometry import Polygon, MultiPolygon

# Period

In [12]:
iyear = 2000
fyear = 2019

# Data for the tab

In [13]:
#dhdt 
dhdt = pd.read_csv('/bettik/PROJECTS/pr-regional-climate/bricej/paper/dhdt_2000-2019_HIMAPgrouped_regions.csv')

#era5 variables
data_era5 = {}
variables = ['t2m','FLH','tcwv','ivt','vimd','tp','sf','rf', 'R']

for var in variables:
    data_var = pd.read_csv(f'/bettik/PROJECTS/pr-regional-climate/bricej/paper/{var}_era5_trends_HIMAPgrouped_regions_{iyear}-{fyear}.csv')
    data_var = data_var[["region", "slope", "pvalue"]]
    data_var["slope"] = data_var["slope"] * 10
    data_era5[var] = data_var

# Making the tab with all data to have the overview

In [14]:
# fonction pour arrondir et ajouter * si pvalue < 0.05
def format_trend(value, pvalue, ndigits=2):
    if pd.isna(value):
        return ""
    txt = f"{value:.{ndigits}f}"
    if pvalue < 0.05:
        txt += "*"
    return txt



# Formater dhdt pour avoir value +/- err
df_dh = dhdt.copy()
df_dh["dhdt_str"] = df_dh.apply(
    lambda x: f"{x['dhdt']:.3f} ± {x['err_dhdt']:.3f}",
    axis=1
)
df_final = df_dh[["region", "dhdt_str"]]



# On ajoute les variables era5 pour avoir le tableau final
# nombre de décimales par variable
decimals = {
    "t2m": 2,
    "tp": 1,
    "sf": 1,
    "rf": 1,
    "tcwv": 2,
    "ivt": 2,
    "vimd": 3,
    "FLH": 0,
    "R": 2
}

for var, df_var in data_era5.items():
    nd = decimals.get(var, 2)  # défaut = 2 décimales
    df_tmp = df_var.copy()
    
    df_tmp[var] = df_tmp.apply(
        lambda x: format_trend(x["slope"], x["pvalue"], nd),
        axis=1
    )
    
    df_tmp = df_tmp[["region", var]]
    
    # merge avec le tableau final
    df_final = df_final.merge(df_tmp, on="region", how="left")

In [15]:
df_final

,region,dhdt_str,t2m,FLH,tcwv,ivt,vimd,tp,sf,rf,R
0,Tien Shan,-0.459 ± 0.030,0.14,33,0.01,-0.09,0.021,-2.2,-0.4,-1.8,0.70
1,Pamir Alay,-0.122 ± 0.038,0.36,21,0.08,-0.86,0.006,-1.0,0.2,-1.3,0.95
2,Pamir,-0.138 ± 0.033,0.48*,25,0.07,-0.21,-0.036,-0.2,0.2,-0.4,0.82
3,Hindu Kush,-0.193 ± 0.035,0.45,-2,0.27*,-0.23,-0.044,3.0,1.7,1.2,0.65
4,Karakoram,-0.073 ± 0.030,0.18,3,0.12,0.81,-0.070,1.4,2.3,-0.9,2.62
5,Kunlun,0.080 ± 0.027,0.31,25,0.07,0.70,0.010,-0.4,-1.0*,0.6,-2.11
6,Spiti Lahaul,-0.383 ± 0.043,0.09,2,0.35*,0.54,-0.212,8.8*,3.5,5.3*,0.08
7,Central Himalaya,-0.477 ± 0.040,0.20,31,0.28,1.88,-0.205,6.0,0.4,5.6,-0.24
8,Bhutan,-0.538 ± 0.051,0.46*,53*,0.24*,1.61,-0.036,0.7,-1.4,2.0,-0.66
9,Nyainqentangla,-0.704 ± 0.062,0.59*,59*,0.17*,1.38,0.024,-0.4,-2.3,1.9,-1.57


# Sub figures per region

## Inner TP

In [40]:
region = 'Inner TP'

In [41]:
# era5 variables trends
slope1 = []
p1 = []

for var in variables : 
    trend = data_era5[var][data_era5[var]['region'] == region][["region", "slope", "pvalue"]]
    slope1.append({
        "region": region,
        "var": var,
        "slope1": trend["slope"].values[0]
    })

    p1.append({
        "region": region,
        "var": var,
        "p1": trend["pvalue"].values[0]
    })

slope1 = pd.DataFrame(slope1)
p1 = pd.DataFrame(p1)

slope1 = slope1.pivot(
    index="region",
    columns="var",
    values="slope1"
).reset_index()
slope1.columns.name = None

p1 = p1.pivot(
    index="region",
    columns="var",
    values="p1"
).reset_index()
p1.columns.name = None

In [47]:
slope1[['FLH', 't2m', 'tp', 'sf', 'rf', 'R']]

,FLH,t2m,tp,sf,rf,R
0,37.246922,0.3871,-0.055807,-0.863064,0.807257,-1.47829
